In [2]:
import os
size = os.path.getsize('/content/LLM (2).zip')
print(f"File size: {size / 1e6:.1f} MB")

File size: 59.8 MB


In [3]:
import os
os.chdir('/content')
print(os.listdir())  # confirm files are there

['.config', 'val.bin', 'train.bin', 'LLM (2).zip', 'medical_tokenizer.json', 'sample_data']


In [4]:
import torch, json

config = {
    "vocab_size": None,
    "block_size": 256,
    "n_embd": 384,
    "n_head": 6,
    "n_layer": 6,
    "dropout": 0.2,
    "batch_size": 32,
    "max_iters": 5000,
    "eval_interval": 500,
    "learning_rate": 3e-4,
    "eval_iters": 200,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

with open("medical_tokenizer.json", "r", encoding="utf-8") as f:
    tokenizer_data = json.load(f)

config["vocab_size"] = len(tokenizer_data["model"]["vocab"])
print(f"Vocab size: {config['vocab_size']}")

train_data = torch.frombuffer(open("train.bin", "rb").read(), dtype=torch.uint16).long()
val_data   = torch.frombuffer(open("val.bin",   "rb").read(), dtype=torch.uint16).long()

print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens:   {len(val_data):,}")
print(f"Device: {config['device']}")

Vocab size: 16000
Train tokens: 23,833,856
Val tokens:   2,671,488
Device: cuda


/tmp/ipykernel_470/2257247844.py:24: UserWarning: The given buffer is not writable, and PyTorch does not support non-writable tensors. This means you can write to the underlying (supposedly non-writable) buffer using the tensor. You may want to copy the buffer to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:1581.)
  train_data = torch.frombuffer(open("train.bin", "rb").read(), dtype=torch.uint16).long()


In [5]:
train_data = torch.frombuffer(bytearray(open("train.bin", "rb").read()), dtype=torch.uint16).long()
val_data   = torch.frombuffer(bytearray(open("val.bin",   "rb").read()), dtype=torch.uint16).long()

In [6]:
def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - config["block_size"], (config["batch_size"],))
    x = torch.stack([data[i   : i + config["block_size"]    ] for i in ix])
    y = torch.stack([data[i+1 : i + config["block_size"] + 1] for i in ix])
    return x.to(config["device"]), y.to(config["device"])

# Quick test
xb, yb = get_batch("train")
print(f"Input shape:  {xb.shape}")
print(f"Target shape: {yb.shape}")
print(f"Device: {xb.device}")


Input shape:  torch.Size([32, 256])
Target shape: torch.Size([32, 256])
Device: cuda:0


In [7]:
import torch.nn as nn
from torch.nn import functional as F

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(config["n_embd"], head_size, bias=False)
        self.query = nn.Linear(config["n_embd"], head_size, bias=False)
        self.value = nn.Linear(config["n_embd"], head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(config["block_size"], config["block_size"])))
        self.dropout = nn.Dropout(config["dropout"])

    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ self.value(x)


class MultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = config["n_embd"] // config["n_head"]
        self.heads = nn.ModuleList([Head(head_size) for _ in range(config["n_head"])])
        self.proj  = nn.Linear(config["n_embd"], config["n_embd"])
        self.dropout = nn.Dropout(config["dropout"])

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config["n_embd"], 4 * config["n_embd"]),
            nn.GELU(),
            nn.Linear(4 * config["n_embd"], config["n_embd"]),
            nn.Dropout(config["dropout"]),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa  = MultiHeadAttention()
        self.ffw = FeedForward()
        self.ln1 = nn.LayerNorm(config["n_embd"])
        self.ln2 = nn.LayerNorm(config["n_embd"])

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffw(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding    = nn.Embedding(config["vocab_size"], config["n_embd"])
        self.position_embedding = nn.Embedding(config["block_size"], config["n_embd"])
        self.blocks  = nn.Sequential(*[Block() for _ in range(config["n_layer"])])
        self.ln_final = nn.LayerNorm(config["n_embd"])
        self.lm_head  = nn.Linear(config["n_embd"], config["vocab_size"])

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=config["device"]))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_final(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -config["block_size"]:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs  = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

print("Architecture defined successfully!")

Architecture defined successfully!


In [8]:
model = GPT().to(config["device"])
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params/1e6:.2f}M")
print(f"Model device: {next(model.parameters()).device}")

Model parameters: 23.04M
Model device: cuda:0


In [9]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(config["eval_iters"])
        for k in range(config["eval_iters"]):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

for step in range(config["max_iters"]):
    if step % config["eval_interval"] == 0:
        losses = estimate_loss()
        print(f"Step {step:5d} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f}")

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Training complete!")

Step     0 | train loss 9.8532 | val loss 9.8519
Step   500 | train loss 5.5599 | val loss 5.5951
Step  1000 | train loss 5.0140 | val loss 5.0480
Step  1500 | train loss 4.6589 | val loss 4.7029
Step  2000 | train loss 4.4345 | val loss 4.4840
Step  2500 | train loss 4.2545 | val loss 4.3253
Step  3000 | train loss 4.1370 | val loss 4.2138
Step  3500 | train loss 4.0238 | val loss 4.1102
Step  4000 | train loss 3.9266 | val loss 4.0334
Step  4500 | train loss 3.8532 | val loss 3.9510
Training complete!


In [10]:
torch.save(model.state_dict(), "model_weights.pt")

In [12]:
model = GPT()
model.load_state_dict(torch.load("model_weights.pt"))
model.to(config["device"])
model.eval()

GPT(
  (token_embedding): Embedding(16000, 384)
  (position_embedding): Embedding(256, 384)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-5): 6 x Head(
            (key): Linear(in_features=384, out_features=64, bias=False)
            (query): Linear(in_features=384, out_features=64, bias=False)
            (value): Linear(in_features=384, out_features=64, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ffw): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=1536, out_features=384, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ln1): LayerNorm((384,), eps=1e-05, elementwise_affine=True

In [13]:
@torch.no_grad()
def evaluate_model():
    losses = []
    for _ in range(200):
        X, Y = get_batch("val")
        _, loss = model(X, Y)
        losses.append(loss.item())
    return sum(losses) / len(losses)

print("Validation loss:", evaluate_model())

Validation loss: 3.8776352977752686


In [14]:
@torch.no_grad()
def token_accuracy():
    correct = 0
    total = 0

    for _ in range(200):
        X, Y = get_batch("val")
        X, Y = X.to(config["device"]), Y.to(config["device"])

        logits, _ = model(X, Y)
        preds = torch.argmax(logits, dim=-1)

        correct += (preds == Y).sum().item()
        total += Y.numel()

    return correct / total

print("Token Accuracy:", token_accuracy())

Token Accuracy: 0.343936767578125


In [16]:
import math
val_loss = 3.8776352977752686
print("Perplexity:", math.exp(val_loss))

Perplexity: 48.30984150524453


In [17]:
import torch, json
from tokenizers import Tokenizer

# load tokenizer
tokenizer = Tokenizer.from_file("medical_tokenizer.json")

# load model
model = GPT()
model.load_state_dict(torch.load("model_weights.pt"))
model.to(config["device"])
model.eval()

GPT(
  (token_embedding): Embedding(16000, 384)
  (position_embedding): Embedding(256, 384)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-5): 6 x Head(
            (key): Linear(in_features=384, out_features=64, bias=False)
            (query): Linear(in_features=384, out_features=64, bias=False)
            (value): Linear(in_features=384, out_features=64, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ffw): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=1536, out_features=384, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ln1): LayerNorm((384,), eps=1e-05, elementwise_affine=True

In [37]:
import torch, os

# Step 1: Check checkpoint keys
checkpoint = torch.load("/content/model_weights.pt", map_location="cpu")
if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys()))

# Step 2: Check all files in /content
print("\nFiles in /content:")
for f in os.listdir("/content"):
    print(f)

# Step 3: Check HuggingFace cache (tokenizers get cached here)
cache_path = "/root/.cache/huggingface/hub"
if os.path.exists(cache_path):
    print("\nCached HF models/tokenizers:")
    for f in os.listdir(cache_path):
        print(f)

Checkpoint keys: ['token_embedding.weight', 'position_embedding.weight', 'blocks.0.sa.heads.0.tril', 'blocks.0.sa.heads.0.key.weight', 'blocks.0.sa.heads.0.query.weight', 'blocks.0.sa.heads.0.value.weight', 'blocks.0.sa.heads.1.tril', 'blocks.0.sa.heads.1.key.weight', 'blocks.0.sa.heads.1.query.weight', 'blocks.0.sa.heads.1.value.weight', 'blocks.0.sa.heads.2.tril', 'blocks.0.sa.heads.2.key.weight', 'blocks.0.sa.heads.2.query.weight', 'blocks.0.sa.heads.2.value.weight', 'blocks.0.sa.heads.3.tril', 'blocks.0.sa.heads.3.key.weight', 'blocks.0.sa.heads.3.query.weight', 'blocks.0.sa.heads.3.value.weight', 'blocks.0.sa.heads.4.tril', 'blocks.0.sa.heads.4.key.weight', 'blocks.0.sa.heads.4.query.weight', 'blocks.0.sa.heads.4.value.weight', 'blocks.0.sa.heads.5.tril', 'blocks.0.sa.heads.5.key.weight', 'blocks.0.sa.heads.5.query.weight', 'blocks.0.sa.heads.5.value.weight', 'blocks.0.sa.proj.weight', 'blocks.0.sa.proj.bias', 'blocks.0.ffw.net.0.weight', 'blocks.0.ffw.net.0.bias', 'blocks.0.ffw.n

In [42]:
import re
import torch
import torch.nn.functional as F
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("/content/medical_tokenizer.json")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── Stop phrases — generation halts at any of these ──────────────
STOP_PHRASES = [
    "### Instruction", "### Question", "### Response",
    "Q :", "A :", "Which of the following", "Which one of",
    "A patient", "A 30 year", "A 45 year", "A 60 year",
    "The correct answer is", "Ans .", "Ref :", "Ref Harrison",
    "# Niacin", "# The", "\n\n\n",
]

def clean_response(text):
    # 1. Cut at first stop phrase
    for phrase in STOP_PHRASES:
        if phrase in text:
            text = text[:text.index(phrase)]

    # 2. Keep only first 2 sentences (more = more drift)
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s for s in sentences if len(s.strip()) > 10]
    text = " ".join(sentences[:2])

    # 3. Remove broken fragments (lines with no real words)
    text = re.sub(r'\s*[#>\*\-]+\s*', ' ', text)

    # 4. Fix spacing around punctuation
    text = re.sub(r'\s([.,;:!?])', r'\1', text)
    text = re.sub(r'\s+', ' ', text)

    # 5. Cut mid-sentence if too long
    if len(text) > 300:
        text = text[:300].rsplit(' ', 1)[0] + "..."

    return text.strip()


def ask(question, template="qa", temperature=0.55, max_new_tokens=150,
        top_k=35, top_p=0.85, repetition_penalty=1.8):

    TEMPLATES = {
        "qa":       "### Instruction:\n{q}\n\n### Response:\n",
        "clinical": "### Question:\n{q}\n\n### Answer:\n",
        "mcqa":     "### Question:\n{q}\n\n### Explanation and Answer:\n",
    }
    prompt = TEMPLATES[template].format(q=question)
    ids    = tokenizer.encode(prompt).ids
    idx    = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    token_counts = {}

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits, _ = model(idx[:, -256:])
            logits     = logits[:, -1, :].clone()

            # Repetition penalty
            for tok_id, count in token_counts.items():
                logits[:, tok_id] /= (repetition_penalty ** min(count, 4))

            logits /= temperature

            # Top-k
            top_k_vals, _ = torch.topk(logits, top_k)
            logits[logits < top_k_vals[:, [-1]]] = float("-inf")

            # Top-p
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            remove    = cum_probs - F.softmax(sorted_logits, dim=-1) > top_p
            sorted_logits[remove] = float("-inf")
            logits = torch.scatter(logits, 1, sorted_idx, sorted_logits)

            probs    = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            tok_id   = next_tok.item()
            token_counts[tok_id] = token_counts.get(tok_id, 0) + 1

            # Hard stop if token repeated 4+ times
            if token_counts[tok_id] >= 4:
                break

            idx = torch.cat([idx, next_tok], dim=1)

    raw      = tokenizer.decode(idx[0][len(ids):].tolist())
    cleaned  = clean_response(raw)
    return cleaned


# ── Test ─────────────────────────────────────────────────────────
tests = [
    "What is Metformin used for?",
    "What are symptoms of diabetes?",
    "What causes hypertension?",
    "What is the treatment for pulmonary embolism?",
]

for q in tests:
    print(f"\nQ: {q}")
    print(f"A: {ask(q)}")
    print("─" * 55)


Q: What is Metformin used for?
A: Glucose: 6 ( 8 )
───────────────────────────────────────────────────────

Q: What are symptoms of diabetes?
A: Loss of the blood and is usually due to increased production. Which one following does not cause pain?
───────────────────────────────────────────────────────

Q: What causes hypertension?
A: Renal failure: Hepatic encephalopathy, biliary cirrhosis and hepatic disease.
───────────────────────────────────────────────────────

Q: What is the treatment for pulmonary embolism?
A: Surgical decompression: Treatment of acute pulmonary embolism is: 1. 5 2 % ( in patients with chronic hepatitis cirrhosis ) or without any myocardial infarction, IV IG and T3. Surgery may be indicated by endoscopic therapy if the patient has a coronary artery surgery can safely require treatment...
───────────────────────────────────────────────────────
